### Products That Never Got Any Order - write pyspark code

In [0]:
# ============================================================================
# POPULAR INTERVIEW QUESTION: "Products That Never Got Any Order"
#
# You have two tables: Products and Orders. Return ONLY the products that
# have NEVER appeared in any order.
#
# This tests your understanding of the ANTI-JOIN pattern — the flip side of
# the INNER JOIN. The two idiomatic solutions are:
#   1. LEFT JOIN + WHERE right key IS NULL   (classic SQL interview answer)
#   2. LEFT ANTI JOIN                          (the Spark-native, cleaner way)
#
# Input:
# PRODUCTS                     ORDERS
# +----+-----------+-------+   +--------+------------+
# | id | name      | price |   | ord_id | product_id |
# +----+-----------+-------+   +--------+------------+
# | 1  | Laptop    | 1200  |   | 501    | 1          |
# | 2  | Mouse     | 25    |   | 502    | 1          |
# | 3  | Keyboard  | 75    |   | 503    | 3          |
# | 4  | Monitor   | 300   |   | 504    | 5          |  <- product_id 5 doesn't exist (orphan)
# | 5  | Webcam    | 60    |   +--------+------------+
# +----+-----------+-------+
#
# Expected output (products with no orders = Laptop? no, it HAS orders 501/502;
# only Mouse and Monitor never got any order):
# +-----------+
# | name      |
# +-----------+
# | Mouse     |
# | Monitor   |
# +-----------+
# ============================================================================


In [0]:
from pyspark.sql import functions as F

# --- Build the sample data ---
products = spark.createDataFrame(
    [
        (1, "Laptop",   1200),
        (2, "Mouse",      25),
        (3, "Keyboard",   75),
        (4, "Monitor",   300),
        (5, "Webcam",     60),
    ],
    ["id", "name", "price"],
)

orders = spark.createDataFrame(
    [
        (501, 1),   
        (502, 1),  
        (503, 3),  
        (504, 5),  
    ],
    ["ord_id", "product_id"],
)

In [0]:
# ============================================================================
# SOLUTION 1: LEFT ANTI JOIN (the Spark-native way)
#
# "left_anti" returns ONLY the rows from the LEFT dataframe that have NO
# match in the right — columns of the right side are not carried at all.
# ============================================================================
result_anti = products.join(
    orders,
    products["id"] == orders["product_id"],
    "left_anti",
).select("id", "name", "price")

result_anti.show()

In [0]:
# ============================================================================
# SOLUTION 2: LEFT JOIN + IS NULL (the classic SQL interview answer)
#
# SQL equivalent:
#   SELECT p.id, p.name, p.price
#   FROM products p
#   LEFT JOIN orders o ON o.product_id = p.id
#   WHERE o.ord_id IS NULL;
#
# Every unmatched product keeps its row, with NULLs on the order side —
# filtering those NULLs out leaves exactly the never-ordered products.
# ============================================================================
result_left = (
    products.alias("p")
    .join(orders.alias("o"), F.col("o.product_id") == F.col("p.id"), "left")
    .filter(F.col("o.ord_id").isNull())
    .select(F.col("p.id"), F.col("p.name"), F.col("p.price"))
)

result_left.show()

In [0]:
# ============================================================================
# BONUS — the common TRAP interviewers watch for:
# "Why not just: WHERE product_id NOT IN (SELECT product_id FROM orders)?"
# In SQL, NOT IN blows up (returns zero rows) if the subquery contains a NULL.
# The LEFT ANTI JOIN / LEFT JOIN + IS NULL patterns are NULL-safe.
#
# Key points interviewers look for:
# 1. You know "find X without Y" = ANTI-JOIN, not a plain join filter.
# 2. You pick LEFT ANTI JOIN in Spark — it's cheaper because Spark skips
#    carrying the right side's columns entirely.
# 3. You can explain why LEFT JOIN + IS NULL works: unmatched left rows
#    survive with NULLs, and the IS NULL filter keeps exactly those.
# ============================================================================